##Week10. PCA 및 모델 성능 비교 실습

활용 데이터셋 (Kaggle) 데이터셋 이름: Red Wine Quality Dataset

Kaggle 링크: https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009

와인의 다양한 화학적 속성(산도, 당도, pH 등)을 바탕으로 와인의 품질(Quality)을 분류하는 데이터셋입니다. CSV 파일을 다운로드하여 winequality-red.csv로 업로드한 뒤 문제 풀이를 진행하시면 됩니다.

In [1]:
#실행해주세요.
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Kaggle에서 다운로드한 CSV 파일을 코랩에 업로드한 후 읽어옵니다.
df = pd.read_csv('winequality-red.csv')

# Target 값(quality)을 제외한 피처 데이터세트와 Target 데이터세트 분리
X_features = df.drop('quality', axis=1)
y_target = df['quality']

print('피처 데이터 shape:', X_features.shape)

피처 데이터 shape: (1599, 11)


#1. 데이터 표준화 (StandardScaler)
Target 값을 제외한 모든 피처 속성 값을 StandardScaler를 이용하여 표준 정규 분포를 가지는 값들로 변환하세요.

In [2]:
# StandardScaler 객체 생성
scaler = StandardScaler()

# 피처 데이터세트에 fit_transform()을 호출하여 데이터 스케일링
df_scaled = scaler.fit_transform(df)

print('스케일링된 데이터 shape:', df_scaled.shape)

스케일링된 데이터 shape: (1599, 12)


## 2. PCA 변환 및 변동성 확인
스케일링된 데이터에 대해 2개의 주성분(Component)을 가진 PCA 객체를 생성하고, 변환을 수행하세요. 이후 각 컴포넌트별 변동성 비율(explained variance ratio)을 출력하세요.

In [5]:
# 2개의 주성분을 가진 PCA 객체 생성
pca = PCA(n_components=2)

# 스케일링된 데이터에 fit()과 transform()을 호출해 PCA 변환 데이터 반환
df_pca = pca.fit_transform(df_scaled)

print('PCA 변환 데이터 shape:', df_pca.shape)
print('PCA Component별 변동성:', pca.explained_variance_ratio_)

PCA 변환 데이터 shape: (1599, 2)
PCA Component별 변동성: [0.26009731 0.1868235 ]


## 3. 원본 데이터와 PCA 변환 데이터의 모델 예측 성능 비교
RandomForestClassifier를 이용해 원본 데이터와 PCA 변환 데이터의 교차 검증(Cross Validation) 평균 정확도를 각각 계산하고 비교하세요. (단, random_state=156, cv=3으로 설정)

In [6]:
# 랜덤 포레스트 분류기 객체 생성
rcf = RandomForestClassifier(random_state=156)

# 1. 원본 데이터 교차 검증
scores_original = cross_val_score(rcf, df_scaled, y_target, scoring='accuracy', cv=3)
print('원본 데이터 교차 검증 개별 정확도:', scores_original)
print('원본 데이터 평균 정확도:', np.mean(scores_original))

# 2. PCA 변환 데이터 교차 검증
scores_pca = cross_val_score(rcf, df_pca, y_target, scoring='accuracy', cv=3)
print('\nPCA 변환 데이터 교차 검증 개별 정확도:', scores_pca)
print('PCA 변환 데이터 평균 정확도:', np.mean(scores_pca))

원본 데이터 교차 검증 개별 정확도: [0.97748593 0.98686679 0.98499062]
원본 데이터 평균 정확도: 0.9831144465290808

PCA 변환 데이터 교차 검증 개별 정확도: [0.59662289 0.64165103 0.60037523]
PCA 변환 데이터 평균 정확도: 0.6128830519074421


# 4. SVD

digits 데이터셋을 활용합니다

## 4-1. Truncated SVD를 이용한 이미지 차원 축소

Digits 데이터의 64개 속성을 TruncatedSVD를 사용하여 10개의 컴포넌트로 축소하는 코드

In [7]:
from sklearn.decomposition import TruncatedSVD
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt

# Digits 데이터 로드
digits = load_digits()
digits_ftrs = digits.data

# 1. 10개의 주요 컴포넌트로 TruncatedSVD 변환 객체 생성
tsvd = TruncatedSVD(n_components=10) # [cite: 185, 199]

# 2. 모델 학습 및 데이터 변환 수행
tsvd.fit(digits_ftrs)
digits_tsvd = tsvd.transform(digits_ftrs) # [cite: 189, 201]

print('원본 데이터 형태:', digits_ftrs.shape)
print('SVD 변환 데이터 형태:', digits_tsvd.shape)

원본 데이터 형태: (1797, 64)
SVD 변환 데이터 형태: (1797, 10)


## 4-2. 스케일링에 따른 PCA와 SVD의 일치성 확인

데이터 세트가 스케일링되어 중심이 동일해지면 PCA와 SVD는 동일한 변환을 수행합니다


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD, PCA

# 1. 데이터를 StandardScaler로 표준화 변환
scaler = StandardScaler()
digits_scaled = scaler.fit_transform(digits_ftrs) # [cite: 241]

# TruncatedSVD와 PCA를 각각 2개의 컴포넌트로 학습/변환
tsvd = TruncatedSVD(n_components=2)
pca = PCA(n_components=2)

digits_tsvd = tsvd.fit_transform(digits_scaled)
digits_pca = pca.fit_transform(digits_scaled)

# 2. 두 변환 행렬 값의 차이 평균 출력 (0에 가까울수록 동일)
print('두 변환의 차이 평균:', (digits_pca - digits_tsvd).mean()) # [cite: 281]

두 변환의 차이 평균: 1.2826190134717442e-16


# 5. NMF

digits 데이터셋을 활용합니다

## 5-1. NMF를 이용한 숫자 이미지 잠재 요소 도출

Digits 데이터는 픽셀 값이 0~16 사이의 양수이므로 NMF 적용이 가능합니다

In [9]:
from sklearn.decomposition import NMF

# 1. 10개의 컴포넌트로 NMF 객체 생성
nmf = NMF(n_components=10) # [cite: 307]

# 2. NMF 학습 및 변환 수행
nmf.fit(digits_ftrs)
digits_nmf = nmf.transform(digits_ftrs) # [cite: 307]

print('NMF 변환 데이터 형태:', digits_nmf.shape)

NMF 변환 데이터 형태: (1797, 10)


/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


## 5-2. NMF 행렬 분해의 구성 요소 이해

NMF는 원본 행렬을 잠재 요소를 포함한 W와 H 행렬로 분해합니다

In [10]:
# 1. 분해 행렬 W: 원본 행에 대해 잠재 요소의 값이 얼마나 되는지 대응하는 행렬
# 학습된 nmf 객체의 transform 결과값임
W = nmf.transform(digits_ftrs)

# 2. 분해 행렬 H: 잠재 요소가 원본 속성(픽셀)을 어떻게 구성하는지 나타내는 행렬
# 사이킷런 NMF 클래스의 특정 속성에 저장됨
H = nmf.components_

print('잠재 요소 구성 행렬(H) 형태:', H.shape) # 출력 결과: (10, 64)

잠재 요소 구성 행렬(H) 형태: (10, 64)


## **6. PCA와 LDA**

wine 데이터 세트의 13개 속성을 2차원으로 축소하여 클래스 분리 성능을 비교하려 합니다. <br>다음 빈칸을 채우고 질문에 답하세요.

In [12]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

wine = load_wine()
wine_scaled = StandardScaler().fit_transform(wine.data)

# 6-1. PCA 변환 수행
pca = PCA(n_components=2)
wine_pca = pca.fit_transform(wine_scaled)

# 6-2. LDA 변환 수행
lda = LinearDiscriminantAnalysis(n_components=2)
lda.fit(wine_scaled, wine.target)
wine_lda = lda.transform(wine_scaled)

print(f"PCA 변환 후 형태: {wine_pca.shape}, LDA 변환 후 형태: {wine_lda.shape}")

PCA 변환 후 형태: (178, 2), LDA 변환 후 형태: (178, 2)


Q 6-3. PCA와 LDA의 결정적인 차이점을 '학습 방식(지도/비지도)'과 '최적화 목표' 관점에서 서술하세요.

답안
- PCA(비지도 학습) : 정답(target label)이 필요 없습니다. 데이터가 어떤 클래스에 속하는지 모른 채, 오직 피처(특성) 데이터의 분포 자체만 보고 차원을 축소합니다

- LDA(지도 학습) : 정답이 반드시 필요합니다. 데이터의 클래스 정보를 알고 있는 상태에서, 각 데이터가 어떤 클래스에 속하는지 분류하기에 가장 좋은 방향을 찾아 차원을 축소합니다

## 7. 매니폴드 학습: **Isomap과 MDS**

굽어 있는 고차원 구조를 평면으로 펼치기 위한 비선형 차원 축소 기법에 대한 이해를 묻는 문제입니다. <br>다음 빈칸을 채우고 O/X 퀴즈를 푸세요.

In [15]:
from sklearn.manifold import MDS, Isomap

# 7-1. 샘플 간의 거리를 최대한 보존하며 저차원 좌표를 찾는 MDS 객체 생성
mds = MDS(n_components=2, random_state=42)
wine_mds = mds.fit_transform(wine_scaled)

print('MDS 변환 전 차원:', wine_scaled.shape)
print('MDS 변환 후 차원:', wine_mds.shape)
print('MDS 변환 데이터 샘플(3개):\n', wine_mds[:3])

# 7-2. 지오데식(Geodesic) 거리를 유지하며 매니폴드를 학습하는 Isomap 객체 생성
isomap = Isomap(n_neighbors=5, n_components=2)
wine_isomap = isomap.fit_transform(wine_scaled)

print('\nIsomap 변환 전 차원:', wine_scaled.shape)
print('Isomap 변환 후 차원:', wine_isomap.shape)
print('Isomap 변환 데이터 샘플(3개):\n', wine_isomap[:3])

MDS 변환 전 차원: (178, 13)
MDS 변환 후 차원: (178, 2)
MDS 변환 데이터 샘플(3개):
 [[-4.21049079 -0.08188048]
 [-2.75345367  1.30862345]
 [-3.24537863  0.30830317]]

Isomap 변환 전 차원: (178, 13)
Isomap 변환 후 차원: (178, 2)
Isomap 변환 데이터 샘플(3개):
 [[-9.36571726 -1.03357979]
 [-4.77346846 -1.6518158 ]
 [-8.86092488 -1.01487306]]


Q 7-3. 다음 설명의 참(O)/거짓(X)을 판단하고, X일 경우 이유를 간단히 적으세요.

1. Isomap은 단순히 유클리드 거리를 사용하여 점 사이의 최단 경로를 구한다. (X)

-> Isomap은 단순히 유클리드 거리가 아니라, 데이터가 이루는 곡면을 따라가는 지오데식(Geodesic)거리를 사용합니다. 이웃한 점들 사이에는 유클리드 거리를 쓰지만, 멀리 떨어진 점들은 이웃들을 거쳐서 가는 그래프 기반의 최단 경로를 계산해 매니폴드 구조를 파악합니다

2. MDS는 전역적인 거리 정보를 유지하는 데 집중하므로 굽어 있는 매니폴드 구조를 펼치는 데 한계가 있을 수 있다. (O)

-> MDS는 모든 데이터 쌍 사이의 직선적 거리를 보존하려고 합니다. 데이터가 스위스 롤처럼 둥글게 말려 있거나 구부러져 있는 경우, 멀리 돌아가는 곡면 거리 대신 뚫고 지나가는 직선 거리를 계산하기 때문에 데이터의 고차원 비선형 구조를 제대로 펼치지 못하는 한계가 있습니다.

## **8. 시각화의 강자: t-SNE의 특징**

t-SNE를 사용하여 고차원 데이터의 군집 구조를 시각화하고, 연산 효율성에 대해 분석하려 합니다. <br>빈칸을 채우고 분석 결과를 서술하세요.

In [13]:
from sklearn.manifold import TSNE

# 8-1. t-SNE 객체 생성 및 변환
tsne = TSNE(n_components=2, random_state=42)
wine_tsne = tsne.fit_transform(wine_scaled)

print('t-SNE 변환 전 차원:', wine_scaled.shape)
print('t-SNE 변환 후 차원:', wine_tsne.shape)
print('t-SNE 변환 데이터 샘플(3개):\n', wine_tsne[:3])

t-SNE 변환 전 차원: (178, 13)
t-SNE 변환 후 차원: (178, 2)
t-SNE 변환 데이터 샘플(3개):
 [[10.311771  -8.546624 ]
 [ 8.054532  -5.55134  ]
 [12.12621   -3.8871381]]


Q 8-2. 연산 속도와 관련된 다음 비교 중 빈칸 (A)에 들어갈 단어를 적으세요.

차원 축소 기법 중 [ (PCA) ]은(는) 시간 복잡도가 O(d)로 매우 빠르지만, <br>t-SNE는 데이터 포인트 간 확률 계산으로 인해 연산 속도가 상대적으로 매우 느립니다.

Q 8-3. t-SNE가 PCA나 MDS에 비해 군집 시각화에서 가지는 강점은 무엇인지 '이웃'이라는 키워드를 포함하여 서술하세요.

답안
- PCA나 MDS는 전역적인 거리 정보(모든 데이터 간의 직선 거리)를 보존하는 데 집중하는 반면, t-SNE는 데이터 포인트 간의 국소적 유사도를 보존하는 데 집중합니다.

- 즉, 고차원에서 가까웠던 이웃 관계에서 저차원에서도 최대한 가깝게 유지하도록 확률적으로 시각화하기 때문에, 비선형 구조를 가진 복잡한 데이터셋에서도 서로 다른 군집들을 뭉개지지 않고 명확하게 분리하여 시각화하는 강점을 가집니다